# Threshold sweep: плотность независимых движений

Ноутбук считает те же метрики плотности, что и основное исследование, но для сетки порогов от `0.2%` до `1.5%` с шагом `0.1%`.

Графики здесь не строятся: результат сохраняется в CSV/Parquet-таблицы для дальнейшего анализа.

In [ ]:
from __future__ import annotations

import io
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import boto3
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from dotenv import load_dotenv
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "build_price_feature_day.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 200)

## Конфигурация

`USE_XRP_REPAIRED_DATASET=True` использует восстановленный parquet, где закрыт крупный разрыв XRP. Если нужен исходный файл, переключите флаг в `False`.

In [ ]:
S3_BUCKET = "binance-data-downloader"
ORIGINAL_S3_KEY = "minute_returns_dataset/features/minute_returns_1m_2020-02-01_2026-02-01.parquet"
XRP_REPAIRED_S3_KEY = "minute_returns_dataset/features/minute_returns_1m_2020-02-01_2026-02-01_xrp_repaired.parquet"

USE_XRP_REPAIRED_DATASET = True
S3_KEY = XRP_REPAIRED_S3_KEY if USE_XRP_REPAIRED_DATASET else ORIGINAL_S3_KEY

CACHE_DIR = PROJECT_ROOT / "analysis" / "cache"
OUTPUT_DIR = CACHE_DIR / "threshold_sweep"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_ORIGINAL_PARQUET = CACHE_DIR / Path(ORIGINAL_S3_KEY).name
LOCAL_XRP_REPAIRED_PARQUET = CACHE_DIR / Path(XRP_REPAIRED_S3_KEY).name
LOCAL_PARQUET = LOCAL_XRP_REPAIRED_PARQUET if USE_XRP_REPAIRED_DATASET else LOCAL_ORIGINAL_PARQUET

SELECTED_SYMBOLS: list[str] | None = None
HORIZONS = list(range(1, 121))
THRESHOLD_PCTS = np.round(np.arange(0.2, 1.5 + 0.0001, 0.1), 1)
THRESHOLDS = (THRESHOLD_PCTS / 100.0).round(6).tolist()
PLATEAU_LEVEL = 0.95

COMPUTE_FULL_SAMPLE = True
COMPUTE_MONTHLY = True
COMPUTE_QUARTERLY = True
MONTHLY_FREQ = "ME"
QUARTERLY_FREQ = "QE"

print("threshold_pct:", THRESHOLD_PCTS.tolist())
print("threshold_decimal:", THRESHOLDS)
print("local parquet:", LOCAL_PARQUET)

## Загрузка данных

In [ ]:
def make_s3_client() -> Any:
    load_dotenv(PROJECT_ROOT / ".env")
    return boto3.client(
        "s3",
        endpoint_url=os.getenv("YC_ENDPOINT"),
        region_name=os.getenv("YC_REGION"),
        aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
        aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
    )


def ensure_local_parquet(bucket: str = S3_BUCKET, key: str = S3_KEY, local_path: Path = LOCAL_PARQUET) -> Path:
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.exists() and local_path.stat().st_size > 0:
        print(f"Using cached parquet: {local_path} ({local_path.stat().st_size / 1024**2:.1f} MiB)")
        return local_path

    print(f"Downloading s3://{bucket}/{key} -> {local_path}")
    s3 = make_s3_client()
    s3.download_file(bucket, key, str(local_path))
    print(f"Downloaded {local_path.stat().st_size / 1024**2:.1f} MiB")
    return local_path


def parquet_columns(path: Path) -> list[str]:
    return pq.ParquetFile(path).schema.names


def symbol_from_return_column(column: str) -> str:
    return column.removesuffix("_return")


def return_column_for_symbol(symbol: str) -> str:
    return symbol if symbol.endswith("_return") else f"{symbol}_return"


local_parquet = ensure_local_parquet()
all_columns = parquet_columns(local_parquet)
timestamp_column = "timestamp"
return_columns = [c for c in all_columns if c != timestamp_column and c.endswith("_return")]

if SELECTED_SYMBOLS is None:
    selected_return_columns = return_columns
else:
    selected_return_columns = [return_column_for_symbol(s) for s in SELECTED_SYMBOLS]
    missing = sorted(set(selected_return_columns) - set(return_columns))
    if missing:
        raise ValueError(f"Missing return columns in parquet: {missing}")

returns = pd.read_parquet(local_parquet, columns=[timestamp_column, *selected_return_columns])
returns[timestamp_column] = pd.to_datetime(returns[timestamp_column], utc=True)
returns = returns.sort_values(timestamp_column).drop_duplicates(timestamp_column).reset_index(drop=True)

print(f"Rows: {len(returns):,}")
print(f"Date range: {returns[timestamp_column].min()} .. {returns[timestamp_column].max()}")
print(f"Symbols: {len(selected_return_columns)}")
display(returns.head())

## Аудит покрытия

Этот блок нужен, чтобы отличать поздний старт монеты от внутренних пропусков. Результаты сохраняются рядом с sweep-таблицами.

In [ ]:
def missingness_report(frame: pd.DataFrame, return_cols: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    gap_rows = []
    total = len(frame)

    for col in return_cols:
        symbol = symbol_from_return_column(col)
        valid = frame[col].notna().to_numpy()
        non_null = int(valid.sum())

        if non_null == 0:
            rows.append(
                {
                    "symbol": symbol,
                    "total_rows": total,
                    "non_null": 0,
                    "null_total": total,
                    "coverage_pct": 0.0,
                    "first_valid": pd.NaT,
                    "last_valid": pd.NaT,
                    "missing_before_first": total,
                    "missing_after_last": 0,
                    "internal_missing": 0,
                    "active_span_minutes": 0,
                    "active_coverage_pct": np.nan,
                }
            )
            continue

        valid_idx = np.flatnonzero(valid)
        first_idx = int(valid_idx[0])
        last_idx = int(valid_idx[-1])
        active = frame[col].iloc[first_idx : last_idx + 1]
        internal_mask = active.isna().to_numpy()
        internal_missing = int(internal_mask.sum())
        active_span = last_idx - first_idx + 1

        rows.append(
            {
                "symbol": symbol,
                "total_rows": total,
                "non_null": non_null,
                "null_total": total - non_null,
                "coverage_pct": non_null / total * 100,
                "first_valid": frame.loc[first_idx, timestamp_column],
                "last_valid": frame.loc[last_idx, timestamp_column],
                "missing_before_first": first_idx,
                "missing_after_last": total - last_idx - 1,
                "internal_missing": internal_missing,
                "active_span_minutes": active_span,
                "active_coverage_pct": non_null / active_span * 100,
            }
        )

        if internal_missing:
            starts = np.flatnonzero(internal_mask & np.r_[True, ~internal_mask[:-1]])
            ends = np.flatnonzero(internal_mask & np.r_[~internal_mask[1:], True])
            for start, end in zip(starts, ends):
                abs_start = first_idx + int(start)
                abs_end = first_idx + int(end)
                gap_rows.append(
                    {
                        "symbol": symbol,
                        "gap_start": frame.loc[abs_start, timestamp_column],
                        "gap_end": frame.loc[abs_end, timestamp_column],
                        "missing_minutes": int(end - start + 1),
                    }
                )

    summary = pd.DataFrame(rows).sort_values(["first_valid", "symbol"], na_position="last")
    gaps = pd.DataFrame(gap_rows)
    if not gaps.empty:
        gaps = gaps.sort_values(["missing_minutes", "gap_start"], ascending=[False, True])
    return summary, gaps


missingness, internal_gaps = missingness_report(returns, selected_return_columns)
missingness.to_csv(OUTPUT_DIR / "missingness_summary.csv", index=False)
internal_gaps.to_csv(OUTPUT_DIR / "internal_gaps.csv", index=False)
display(missingness)

## Расчет плотности

In [ ]:
@dataclass(frozen=True)
class DensityPoint:
    symbol: str
    threshold: float
    threshold_pct: float
    horizon: int
    independent_events: int
    possible_observations: int
    density_per_day: float
    event_share: float


def forward_compound_return(minute_returns: pd.Series, horizon: int) -> pd.Series:
    values = minute_returns.astype("float64").replace([np.inf, -np.inf], np.nan)
    log_values = np.log1p(values)
    rolled = log_values.shift(-1).rolling(window=horizon, min_periods=horizon).sum().shift(-(horizon - 1))
    return np.expm1(rolled)


def count_non_overlapping_events(mask: np.ndarray, horizon: int) -> int:
    starts = np.flatnonzero(mask)
    count = 0
    blocked_until = -1
    for start in starts:
        if start >= blocked_until:
            count += 1
            blocked_until = start + horizon
    return count


def density_curve_for_series(
    minute_returns: pd.Series,
    symbol: str,
    horizons: list[int] = HORIZONS,
    thresholds: list[float] = THRESHOLDS,
) -> pd.DataFrame:
    points: list[DensityPoint] = []
    for h in horizons:
        future = forward_compound_return(minute_returns, h)
        possible = int(future.notna().sum())
        possible_days = possible / 1440.0
        abs_future = future.abs().to_numpy()
        finite = np.isfinite(abs_future)

        for threshold in thresholds:
            mask = finite & (abs_future >= threshold)
            events = count_non_overlapping_events(mask, h)
            points.append(
                DensityPoint(
                    symbol=symbol,
                    threshold=threshold,
                    threshold_pct=round(threshold * 100, 1),
                    horizon=h,
                    independent_events=events,
                    possible_observations=possible,
                    density_per_day=events / possible_days if possible_days else np.nan,
                    event_share=events / possible if possible else np.nan,
                )
            )
    return pd.DataFrame(points)


def build_density_curves(frame: pd.DataFrame, return_cols: list[str]) -> pd.DataFrame:
    curves = []
    for col in return_cols:
        symbol = symbol_from_return_column(col)
        print(f"Computing density curves for {symbol}...")
        curves.append(density_curve_for_series(frame[col], symbol=symbol))
    return pd.concat(curves, ignore_index=True) if curves else pd.DataFrame()

## Summary helpers

In [ ]:
def summarize_curve(group: pd.DataFrame, plateau_level: float = PLATEAU_LEVEL) -> pd.Series:
    ordered = group.sort_values("horizon")
    valid_density = ordered["density_per_day"].dropna()
    if valid_density.empty:
        return pd.Series(
            {
                "best_horizon": np.nan,
                "max_density_per_day": np.nan,
                "mean_density_per_day": np.nan,
                "max_to_mean": np.nan,
                "neighbor_advantage": np.nan,
                "plateau_min_h": np.nan,
                "plateau_max_h": np.nan,
                "plateau_width": np.nan,
            }
        )

    idx = valid_density.idxmax()
    best = ordered.loc[idx]
    max_density = float(best["density_per_day"])
    plateau = ordered.loc[ordered["density_per_day"] >= plateau_level * max_density, "horizon"]
    neighbor = ordered.loc[ordered["horizon"].between(best["horizon"] - 2, best["horizon"] + 2)]
    neighbor_without_best = neighbor.loc[neighbor["horizon"] != best["horizon"]]
    neighbor_mean = neighbor_without_best["density_per_day"].mean()
    neighbor_advantage = max_density / neighbor_mean - 1.0 if len(neighbor_without_best) and neighbor_mean > 0 else np.nan

    return pd.Series(
        {
            "best_horizon": int(best["horizon"]),
            "max_density_per_day": max_density,
            "mean_density_per_day": float(valid_density.mean()),
            "max_to_mean": max_density / valid_density.mean(),
            "neighbor_advantage": neighbor_advantage,
            "plateau_min_h": int(plateau.min()),
            "plateau_max_h": int(plateau.max()),
            "plateau_width": int(plateau.max() - plateau.min() + 1),
        }
    )


def summarize_curves_by(frame: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    rows = []
    for key_values, group in frame.groupby(keys, dropna=False):
        if not isinstance(key_values, tuple):
            key_values = (key_values,)
        row = dict(zip(keys, key_values))
        row.update(summarize_curve(group).to_dict())
        rows.append(row)
    return pd.DataFrame(rows)


def normalize_pandas_freq(freq: str) -> str:
    return {"M": "ME", "Q": "QE"}.get(freq, freq)


def period_density_curves(
    frame: pd.DataFrame,
    return_cols: list[str],
    freq: str,
) -> pd.DataFrame:
    result = []
    periods = frame.set_index(timestamp_column).groupby(pd.Grouper(freq=normalize_pandas_freq(freq)))
    for period_end, period_frame in periods:
        if period_frame.empty:
            continue
        period_frame = period_frame.reset_index()
        period_start = period_frame[timestamp_column].min()
        period_label = f"{period_start.date()}..{period_frame[timestamp_column].max().date()}"
        print(f"Period {period_label}: rows={len(period_frame):,}")
        curves = build_density_curves(period_frame, return_cols)
        curves["period_start"] = period_start
        curves["period_end"] = period_frame[timestamp_column].max()
        curves["period_label"] = period_label
        result.append(curves)
    return pd.concat(result, ignore_index=True) if result else pd.DataFrame()

## Full-sample density sweep

In [ ]:
if COMPUTE_FULL_SAMPLE:
    density_full = build_density_curves(returns, selected_return_columns)
    summary_full = summarize_curves_by(density_full, ["symbol", "threshold", "threshold_pct"])

    density_full.to_parquet(OUTPUT_DIR / "density_full.parquet", index=False, compression="zstd")
    density_full.to_csv(OUTPUT_DIR / "density_full.csv", index=False)
    summary_full.to_csv(OUTPUT_DIR / "summary_full.csv", index=False)

    print(f"density_full rows: {len(density_full):,}")
    print(f"summary_full rows: {len(summary_full):,}")
    display(summary_full.sort_values(["symbol", "threshold_pct"]).head(30))
else:
    density_full = pd.read_parquet(OUTPUT_DIR / "density_full.parquet")
    summary_full = pd.read_csv(OUTPUT_DIR / "summary_full.csv")

## Monthly density sweep

In [ ]:
if COMPUTE_MONTHLY:
    density_monthly = period_density_curves(returns, selected_return_columns, freq=MONTHLY_FREQ)
    summary_monthly = summarize_curves_by(
        density_monthly,
        ["period_label", "period_start", "period_end", "symbol", "threshold", "threshold_pct"],
    )
    summary_monthly["month"] = pd.to_datetime(summary_monthly["period_start"]).dt.to_period("M").astype(str)

    density_monthly.to_parquet(OUTPUT_DIR / "density_monthly.parquet", index=False, compression="zstd")
    summary_monthly.to_csv(OUTPUT_DIR / "summary_monthly.csv", index=False)

    print(f"density_monthly rows: {len(density_monthly):,}")
    print(f"summary_monthly rows: {len(summary_monthly):,}")
    display(summary_monthly.sort_values(["symbol", "threshold_pct", "period_start"]).head(30))
else:
    density_monthly = pd.read_parquet(OUTPUT_DIR / "density_monthly.parquet")
    summary_monthly = pd.read_csv(OUTPUT_DIR / "summary_monthly.csv", parse_dates=["period_start", "period_end"])

## Quarterly density sweep

In [ ]:
if COMPUTE_QUARTERLY:
    density_quarterly = period_density_curves(returns, selected_return_columns, freq=QUARTERLY_FREQ)
    summary_quarterly = summarize_curves_by(
        density_quarterly,
        ["period_label", "period_start", "period_end", "symbol", "threshold", "threshold_pct"],
    )
    summary_quarterly["quarter"] = pd.to_datetime(summary_quarterly["period_start"]).dt.to_period("Q").astype(str)

    density_quarterly.to_parquet(OUTPUT_DIR / "density_quarterly.parquet", index=False, compression="zstd")
    summary_quarterly.to_csv(OUTPUT_DIR / "summary_quarterly.csv", index=False)

    print(f"density_quarterly rows: {len(density_quarterly):,}")
    print(f"summary_quarterly rows: {len(summary_quarterly):,}")
    display(summary_quarterly.sort_values(["symbol", "threshold_pct", "period_start"]).head(30))
else:
    density_quarterly = pd.read_parquet(OUTPUT_DIR / "density_quarterly.parquet")
    summary_quarterly = pd.read_csv(OUTPUT_DIR / "summary_quarterly.csv", parse_dates=["period_start", "period_end"])

## Сводные таблицы по порогам

Ниже сохраняются широкие таблицы, удобные для быстрого сравнения оптимального горизонта, ширины 95%-плато и максимальной плотности по порогам.

In [ ]:
best_horizon_by_threshold = summary_full.pivot(index="symbol", columns="threshold_pct", values="best_horizon").sort_index()
plateau_width_by_threshold = summary_full.pivot(index="symbol", columns="threshold_pct", values="plateau_width").sort_index()
max_density_by_threshold = summary_full.pivot(index="symbol", columns="threshold_pct", values="max_density_per_day").sort_index()

best_horizon_by_threshold.to_csv(OUTPUT_DIR / "wide_best_horizon_by_threshold.csv")
plateau_width_by_threshold.to_csv(OUTPUT_DIR / "wide_plateau_width_by_threshold.csv")
max_density_by_threshold.to_csv(OUTPUT_DIR / "wide_max_density_by_threshold.csv")

print("best_horizon_by_threshold")
display(best_horizon_by_threshold)
print("plateau_width_by_threshold")
display(plateau_width_by_threshold)
print("max_density_by_threshold")
display(max_density_by_threshold)

## Output files

Все результаты лежат в `analysis/cache/threshold_sweep/`:

- `density_full.parquet`, `density_full.csv`
- `summary_full.csv`
- `density_monthly.parquet`, `summary_monthly.csv`
- `density_quarterly.parquet`, `summary_quarterly.csv`
- `wide_best_horizon_by_threshold.csv`
- `wide_plateau_width_by_threshold.csv`
- `wide_max_density_by_threshold.csv`
- `missingness_summary.csv`, `internal_gaps.csv`